In [1]:
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# ============================================================
# CBSE Class 12 Result Extractor - Google Colab  [v2 - FIXED]
# ============================================================

# ── STEP 1: Install dependencies ──────────────────────────
!pip install playwright -q
!playwright install chromium
!playwright install-deps chromium
!apt-get install -y libgbm1 libasound2 -q

# ── STEP 2: Imports & Setup ────────────────────────────────
import asyncio, os, glob
from playwright.async_api import async_playwright
from IPython.display import display, Image, FileLink


# ##############can break code from here################################
# # ── Credentials (all 4 students) ──────────────────────────
# credentials_list = [
#     {"roll_no": "22687719", "school_no": "65921", "admit_id": "IZ196543"},
#     {"roll_no": "22687772", "school_no": "65921", "admit_id": "AN726543"},
#     {"roll_no": "22687735", "school_no": "65921", "admit_id": "AR356543"},
#     {"roll_no": "22687747", "school_no": "65921", "admit_id": "IA476543"},
# ]

# TARGET_URL = "https://examinationservices.nic.in/cbseresults/class_xii_b_2026_a/ClassTwelfth_ii26.htm"
# OUTPUT_DIR = "/content/cbse_results"
# os.makedirs(OUTPUT_DIR, exist_ok=True)


# --- USER CONFIGURATION ---
# Path based on image_2a5215.png
INPUT_CSV_PATH = "/content/drive/MyDrive/CBSE_2026_XII/StudentCredentials.csv"
OUTPUT_DRIVE_DIR = "/content/drive/MyDrive/CBSE_2026_XII/Results_Output123"
TARGET_URL = "https://examinationservices.nic.in/cbseresults/class_xii_b_2026_a/ClassTwelfth_ii26.htm"
OUTPUT_DIR = "/content/cbse_results"
os.makedirs(OUTPUT_DRIVE_DIR, exist_ok=True)

# Load credentials from CSV
df_input = pd.read_csv(INPUT_CSV_PATH)
credentials_list = df_input.to_dict('records')


# ── Core Function ──────────────────────────────────────────
async def fetch_result(page, cred: dict) -> dict:
    roll   = cred["roll_no"]
    school = cred["school_no"]
    admit  = cred["admit_id"]

    print(f"\n{'='*55}")
    print(f"📥  Roll: {roll} | School: {school} | Admit: {admit}")
    print(f"{'='*55}")

    await page.goto(TARGET_URL, wait_until="domcontentloaded", timeout=60_000)
    await page.wait_for_timeout(2000)

    # ── Positional filling using page.locator (not ElementHandle) ──
    # This avoids the triple_click issue entirely
    all_inputs = page.locator("input[type='text'], input:not([type])")
    count = await all_inputs.count()
    print(f"  🔍 Found {count} text input(s)")

    if count < 3:
        raise ValueError(f"Expected ≥3 inputs, found {count}")

    # Clear and fill each field positionally
    await all_inputs.nth(0).click()
    await all_inputs.nth(0).fill("")
    await all_inputs.nth(0).fill(roll)
    await page.wait_for_timeout(200)

    await all_inputs.nth(1).click()
    await all_inputs.nth(1).fill("")
    await all_inputs.nth(1).fill(school)
    await page.wait_for_timeout(200)

    await all_inputs.nth(2).click()
    await all_inputs.nth(2).fill("")
    await all_inputs.nth(2).fill(admit)
    await page.wait_for_timeout(200)

    print(f"  ✅ Fields filled: [{roll}] [{school}] [{admit}]")

    # ── Submit ────────────────────────────────────────────
    submit_selectors = [
        "input[type='submit']",
        "button[type='submit']",
        "input[type='button']",
        "button",
    ]
    submitted = False
    for sel in submit_selectors:
        btn = page.locator(sel).first
        if await btn.count() > 0:
            await btn.click()
            submitted = True
            print(f"  ✅ Submitted via: {sel}")
            break

    if not submitted:
        raise RuntimeError("Could not find a submit button.")

    await page.wait_for_timeout(4000)

    # ── Click Print link if present ───────────────────────
    print_selectors = [
        "a[href*='print' i]",
        "a[onclick*='print' i]",
        "input[value*='print' i]",
        "a:text-matches('print', 'i')",
        "button:text-matches('print', 'i')",
    ]
    for sel in print_selectors:
        try:
            el = page.locator(sel).first
            if await el.count() > 0:
                await el.click()
                await page.wait_for_timeout(1500)
                print(f"  🖨️  Print link clicked: {sel}")
                break
        except Exception:
            continue

    # ── Save PNG screenshot ───────────────────────────────
    png_path = os.path.join(OUTPUT_DIR, f"result_{roll}.png")
    await page.screenshot(path=png_path, full_page=True)
    print(f"  📸 PNG  → {png_path}")

    # ── Save PDF ──────────────────────────────────────────
    pdf_path = os.path.join(OUTPUT_DIR, f"result_{roll}.pdf")
    await page.pdf(
        path=pdf_path,
        format="A4",
        print_background=True,
        margin={"top": "10mm", "bottom": "10mm", "left": "10mm", "right": "10mm"}
    )
    print(f"  💾 PDF  → {pdf_path}")

    return {"roll_no": roll, "png": png_path, "pdf": pdf_path, "status": "success"}


# --- Modification inside main() loop ---
# Ensure the fetch_result function uses the new OUTPUT_DRIVE_DIR
# Change this line in your fetch_result function:
# png_path = os.path.join(OUTPUT_DRIVE_DIR, f"result_{roll}.png")
# pdf_path = os.path.join(OUTPUT_DRIVE_DIR, f"result_{roll}.pdf")

async def main():
    results_log = []
    async with async_playwright() as pw:
        # ... (browser setup code same as your working version)

        for cred in credentials_list:
            try:
                # We pass OUTPUT_DRIVE_DIR here
                result = await fetch_result(page, cred)
                results_log.append(result)
            except Exception as e:
                results_log.append({
                    "roll_no": cred['roll_no'],
                    "status": "failed",
                    "error": str(e),
                    "pdf": None
                })
        # ...
    return results_log

# ── RUN ────────────────────────────────────────────────────
results = await main()


# ============================================================
# STEP 3 — Show PNGs inline + sidebar links
# ============================================================
print("\n╔══════════════════════════════════════════╗")
print("║        📊  EXTRACTION SUMMARY            ║")
print("╚══════════════════════════════════════════╝")
for r in results:
    status = "✅ SUCCESS" if r["status"] == "success" else f"❌ FAILED: {r.get('error','?')}"
    print(f"  Roll {r['roll_no']} → {status}")

# ── Inline PNG previews ───────────────────────────────────
# print("\n\n🖼️  Result Screenshots:\n")
# for r in results:
#     if r["status"] == "success":
#         print(f"\n── Roll No: {r['roll_no']} ──────────────────────────")
#         display(Image(r["png"], width=750))

# # ── Per-file download links ───────────────────────────────
# print("\n\n📥  PNG Download Links:\n")
# for r in results:
#     if r["status"] == "success":
#         display(FileLink(r["png"],  result_html_prefix=f"🖼️  Roll {r['roll_no']} PNG: "))

# print("\n📥  PDF Download Links:\n")
# for r in results:
#     if r["status"] == "success":
#         display(FileLink(r["pdf"], result_html_prefix=f"📄  Roll {r['roll_no']} PDF: "))

# ── Bulk ZIP ──────────────────────────────────────────────
print("\n\n📦  Creating ZIP of all results…")
!zip -j /content/cbse_all_results.zip {OUTPUT_DIR}/*.png {OUTPUT_DIR}/*.pdf 2>/dev/null
print("✅  Zip ready:")
display(FileLink("/content/cbse_all_results.zip", result_html_prefix="⬇️  Download ALL (ZIP): "))

print(f"\n📁  Sidebar path: {OUTPUT_DIR}")
print("    Click the 📁 folder icon (left panel) → /content/cbse_results")


Installing dependencies...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1

/content/cbse_all_results.zip


📁  Sidebar path: /content/cbse_results
    Click the 📁 folder icon (left panel) → /content/cbse_results


In [5]:
# ── STEP 1: Imports ──────────────────────────────────────────
import asyncio, os
import pandas as pd
from google.colab import drive
from playwright.async_api import async_playwright

# ── STEP 2: Mount Drive & Configuration ──────────────────────
drive.mount('/content/drive')

# Paths based on your Drive structure (image_2a5215.png)
INPUT_CSV_PATH = "/content/drive/MyDrive/CBSE_2026_XII/StudentCredentials.csv"
OUTPUT_DRIVE_DIR = "/content/drive/MyDrive/CBSE_2026_XII/Results_Output"
TARGET_URL = "https://examinationservices.nic.in/cbseresults/class_xii_b_2026_a/ClassTwelfth_ii26.htm"

os.makedirs(OUTPUT_DRIVE_DIR, exist_ok=True)

# Load credentials from your spreadsheet-generated CSV
df_input = pd.read_csv(INPUT_CSV_PATH)
credentials_list = df_input.to_dict('records')

# ── STEP 3: Core Extraction Function ──────────────────────────
async def fetch_result(page, cred: dict):
    roll   = str(cred["roll_no"])
    school = str(cred["school_no"])
    admit  = str(cred["admit_id"])

    print(f"📥 Fetching PDF for Roll: {roll}...")

    await page.goto(TARGET_URL, wait_until="domcontentloaded", timeout=60_000)
    await page.wait_for_timeout(2000)

    # Filling input fields positionally
    all_inputs = page.locator("input[type='text'], input:not([type])")
    if await all_inputs.count() < 3:
        raise ValueError("Input fields not detected on page.")

    await all_inputs.nth(0).fill(roll)
    await all_inputs.nth(1).fill(school)
    await all_inputs.nth(2).fill(admit)

    # Click Submit
    submit_btn = page.locator("input[type='submit'], button[type='submit']").first
    await submit_btn.click()
    await page.wait_for_timeout(4000)

    # Save PDF directly to your Drive folder
    pdf_path = os.path.join(OUTPUT_DRIVE_DIR, f"result_{roll}.pdf")
    await page.pdf(
        path=pdf_path,
        format="A4",
        print_background=True,
        margin={"top": "10mm", "bottom": "10mm", "left": "10mm", "right": "10mm"}
    )
    print(f"✅ Uploaded to Drive: result_{roll}.pdf")

# ── STEP 4: Main Execution Runner ─────────────────────────────
async def run_extraction():
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True, args=["--no-sandbox"])
        context = await browser.new_context()
        page    = await context.new_page()

        for cred in credentials_list:
            try:
                await fetch_result(page, cred)
            except Exception as e:
                print(f"❌ Failed to process Roll {cred.get('roll_no')}: {e}")

        await browser.close()

# Start the process
await run_extraction()
print(f"\n🚀 All processing complete. PDFs are in: {OUTPUT_DRIVE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 Fetching PDF for Roll: 22687719...
✅ Uploaded to Drive: result_22687719.pdf
📥 Fetching PDF for Roll: 22687772...
✅ Uploaded to Drive: result_22687772.pdf
📥 Fetching PDF for Roll: 22687735...
✅ Uploaded to Drive: result_22687735.pdf
📥 Fetching PDF for Roll: 22687747...
✅ Uploaded to Drive: result_22687747.pdf

🚀 All processing complete. PDFs are in: /content/drive/MyDrive/CBSE_2026_XII/Results_Output
